# Intro to Scaled Qualitative Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Corey-Abramson/Intro-to-Scaled-Qualitative-Analysis/blob/main/notebook/intro_scaled_qualitative_analysis.ipynb)

**What this is.** A teaching notebook compiled for the [Institute on AI Methods
for Social Scientists (AIMS)](https://casbs.stanford.edu/programs/summer-institutes/institute-ai-methods-social-scientists-aims)
at the Center for Advanced Study in the Behavioral Sciences, Stanford
University. It follows the approach set out in Abramson, Prendergast, Li and
Dohan (2026) and draws on the lab's public teaching repositories, linked below.

Interview transcripts arrive as text. Analysing them at scale needs rows. This
notebook runs that conversion end to end on public interview data, then codes
the result three different ways and draws two figures from the codes it made.

**normalize → read into a table → classify → visualize**

| Stage | What you will do |
|---|---|
| **1. Normalize** | Turn one messy raw transcript into machine-readable rows |
| **2. Read into a table** | Load a coded corpus that already uses that format |
| **3. Classify** | Code text three ways: a dictionary, a model, and a language model |
| **4. Visualize** | Draw a word cloud and a code co-occurrence heatmap |

Run the cells top to bottom. It works locally and in Colab. There are no API
keys, no model downloads, and nothing to configure.

**New to any of this?** Short introductions to
[Jupyter notebooks](https://docs.jupyter.org/en/latest/start/index.html),
[Google Colab](https://colab.research.google.com/notebooks/intro.ipynb), and
[GitHub](https://docs.github.com/en/get-started/start-your-journey/hello-world).
There is also a very useful [video by Rice grad Jakira
Silas](https://vimeo.com/1122226315) introducing how to use the toolkit in
Google Colab. Highly recommended if you have not used Python, Jupyter, or Colab
before. The lab's teaching repository collects the rest under
[A Few Key Links](https://github.com/Computational-Ethnography-Lab/teaching#a-few-key-links).

**Getting your own data in.** The
[CMAP QDPX Converter](https://github.com/Computational-Ethnography-Lab/cmap_qdpx_converter)
gets data out of ATLAS.ti, NVivo, or MAXQDA and into the format used here. The
[CMAP Visualization Toolkit](https://github.com/Computational-Ethnography-Lab/cmap_visualization_toolkit)
has the fuller versions of the figures this notebook draws, and the code in
Stage 4 is recycled from it.

**Licensing.** The code here is BSD 3-Clause. The dataset is **not**. It keeps
the IEEE History Center's own restriction on quotation. See
[`SAMPLE_DATA.md`](../SAMPLE_DATA.md).

**References.** Rather than a wall of citations here, see the lab's curated
[topical bibliography](https://github.com/Computational-Ethnography-Lab/teaching#v-bibliography)
and the [Computational Ethnography Lab](https://computationalethnography.org/).
Works cited in this notebook are listed in [`REFERENCES.md`](../REFERENCES.md),
and the closing section repeats the load-bearing ones in full.

**Disclosure.** This teaching repository was compiled and adapted from existing
workflows, based on collaborations and public repositories, using Opus 5.0 in
Claude Code and GPT Sol. Please look at the original code it was adapted from.
Colab runs online: sensitive data belongs on a local machine or in a secure,
IRB-approved environment. This is a demonstration for teaching, not production
software. No private information or data is used here.


In [ ]:
# Setup. Finds the repository, installs what is missing, checks versions.

import os
import subprocess
import sys
from pathlib import Path

RELEASE_TAG = "v1.0.0"
REPO_URL = "https://github.com/Corey-Abramson/Intro-to-Scaled-Qualitative-Analysis.git"
SENTINEL = Path("cmap_demo") / "__init__.py"

# Everything the notebook actually needs. Checking the sentinel alone is not
# enough: an unrelated cmap_demo/__init__.py anywhere above the working
# directory would match, we would chdir into it, and the very next import
# would fail with a confusing ModuleNotFoundError. Worse, in Colab that wrong
# match would preempt the clone branch, so the correct fallback never runs.
MUST_EXIST = [
    SENTINEL,
    Path("cmap_demo") / "viz.py",
    Path("cmap_demo") / "normalize.py",
    Path("cmap_demo") / "header.py",
    Path("cmap_demo") / "llm_handoff.py",
    Path("data") / "1_cleaned_data.csv",
]

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


def find_repo_root(start):
    """Walk up from `start` for a directory holding a COMPLETE checkout."""
    for candidate in [start, *start.parents]:
        if all((candidate / required).exists() for required in MUST_EXIST):
            return candidate
    return None


root = find_repo_root(Path.cwd().resolve())

if root is not None:
    # Branch A: the repository is already on disk. Nothing to fetch.
    os.chdir(root)
    print(f"[OK] Found the repository at {root}")
elif IN_COLAB:
    # Branch B: opened from the Colab badge, which brings only this notebook.
    # Clone the pinned release next to it.
    target = Path("/content") / f"Intro-to-Scaled-Qualitative-Analysis-{RELEASE_TAG}"
    if not (target / SENTINEL).exists():
        print(f"[..] Cloning {RELEASE_TAG} ...")
        clone = subprocess.run(
            ["git", "clone", "--branch", RELEASE_TAG, "--depth", "1",
             REPO_URL, str(target)],
            capture_output=True, text=True,
        )
        if clone.returncode != 0:
            raise RuntimeError(
                f"Could not clone {RELEASE_TAG} from {REPO_URL}. "
                f"git said: {clone.stderr.strip()} "
                "Usually this means the release tag is not published yet, or "
                "this runtime has no network access. You can still run the "
                "notebook: download the repository as a zip, upload it to "
                "this Colab session, unzip it, and re-run this cell from "
                "inside the unzipped folder."
            )
    os.chdir(target)
    checked_out = subprocess.run(
        ["git", "describe", "--tags", "--exact-match"],
        capture_output=True, text=True,
    ).stdout.strip()
    if checked_out != RELEASE_TAG:
        raise RuntimeError(
            f"Expected release {RELEASE_TAG} but the checkout reports "
            f"{checked_out!r}. Stopping rather than running unknown code."
        )
    root = target
    print(f"[OK] Cloned {RELEASE_TAG} to {root}")
else:
    raise RuntimeError(
        "Could not find cmap_demo/__init__.py by walking up from "
        f"{Path.cwd()}. Run this notebook from inside a clone of the "
        "repository, or open it with the Colab badge."
    )

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

# Check what is present, and whether it actually works. On Colab the
# preinstalled numpy/pandas/scipy are left alone: forcing a downgrade there is
# slow and breaks other things.
REQUIRED = {
    "pandas": "pandas", "numpy": "numpy", "matplotlib": "matplotlib",
    "seaborn": "seaborn", "scipy": "scipy", "wordcloud": "wordcloud",
    "PIL": "pillow",
}

ENV_ADVICE = """
This is an environment problem, not a problem with the notebook. Any one of
these fixes it:

  1. If you installed anything in this session, RESTART THE KERNEL and run
     this cell again. Upgrading a compiled package underneath a running
     kernel leaves it half loaded.

  2. Update the whole set together, so they agree with each other:
     python -m pip install -U numpy pandas matplotlib seaborn scipy wordcloud pillow

  3. Cleanest, and what to do before a talk: build a fresh environment and
     select it as the kernel.
     python -m venv .venv
     .venv/bin/pip install -r requirements.txt ipykernel
     .venv/bin/python -m ipykernel install --user --name intro-scaled-qual
"""


def probe_packages():
    """Return (missing, broken). Broken means present but not usable."""
    missing, broken = [], []
    for module_name, package_name in REQUIRED.items():
        try:
            __import__(module_name)
        except ImportError:
            missing.append(package_name)
        except Exception as exc:
            # A package compiled against a different NumPy imports and then
            # dies on the C API, typically "_ARRAY_API not found". That is an
            # AttributeError, not an ImportError, so it must be caught here or
            # it escapes as an unreadable traceback.
            broken.append(f"{package_name} ({type(exc).__name__}: {exc})")
    return missing, broken


def check_binary_compatibility():
    """Exercise the compiled paths, so a mismatch surfaces here and not later.

    Version numbers cannot detect this. A package built against NumPy 1.x sits
    happily beside NumPy 2.x until something touches the C API.
    """
    import numpy as _np
    import pandas as _pd
    from matplotlib.transforms import Affine2D

    sample = _np.arange(6.0).reshape(3, 2)
    Affine2D().transform(sample)
    _pd.DataFrame(sample).sum().sum()


missing, broken = probe_packages()

if broken:
    raise RuntimeError(
        "These packages are installed but not usable in this kernel: "
        + "; ".join(broken) + ENV_ADVICE
    )

if missing:
    print(f"[..] Installing {', '.join(missing)} ...")
    install = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", *missing],
        capture_output=True, text=True,
    )
    if install.returncode != 0:
        raise RuntimeError(
            f"Could not install: {', '.join(missing)}. "
            f"pip said: {install.stderr.strip()[-800:]} "
            "If this machine is offline or behind a proxy, install them "
            "beforehand with: pip install -r requirements.txt"
        )
    print("[OK] Installed")

    # Installing can pull in a newer NumPy underneath packages that were
    # compiled for an older one. Re-probe rather than assume it went well.
    missing, broken = probe_packages()
    if broken:
        raise RuntimeError(
            "Installing " + ", ".join(missing or ["those packages"])
            + " left this kernel inconsistent: " + "; ".join(broken)
            + ". RESTART THE KERNEL and run this cell again."
            + ENV_ADVICE
        )
else:
    print("[OK] All required packages already present")

try:
    check_binary_compatibility()
except Exception as exc:
    import numpy as _np_report
    raise RuntimeError(
        f"The compiled packages in this kernel disagree with NumPy "
        f"{_np_report.__version__}: {type(exc).__name__}: {exc}"
        + ENV_ADVICE
    ) from exc
print("[OK] Compiled packages agree with the installed NumPy")

# Version gate. These bounds were verified by running this whole workflow on
# numpy 1.26.4 / pandas 2.1.4, numpy 2.2.6 / pandas 2.2.3, and
# numpy 2.4.6 / pandas 3.0.5. Outside them, stop rather than fail mid-figure.
import numpy as np
import pandas as pd


def _major_minor(version):
    parts = version.split(".")
    return int(parts[0]), int(parts[1])


for name, version, low, high in [
    ("numpy", np.__version__, (1, 26), (3, 0)),
    ("pandas", pd.__version__, (2, 1), (4, 0)),
]:
    if not (low <= _major_minor(version) < high):
        raise RuntimeError(
            f"{name} {version} is outside the tested range "
            f"{low[0]}.{low[1]} to below {high[0]}.{high[1]}. "
            f"Install a supported version with: pip install -r requirements.txt"
        )
print(f"[OK] numpy {np.__version__}, pandas {pd.__version__} are in the tested range")

# Figures and CSVs land here. Git does not keep an empty ignored directory, so
# a fresh clone has no output/ until this runs.
OUTPUT_DIR = Path("output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# NLTK is optional. If these downloads fail or are blocked, the word cloud
# falls back to a bundled stop-word list and a regex tokenizer.
try:
    import nltk
    for resource in ["punkt", "punkt_tab", "stopwords"]:
        try:
            nltk.download(resource, quiet=True)
        except Exception:
            pass
    print("[OK] NLTK data requested (optional)")
except ImportError:
    print("[--] NLTK not installed; the bundled fallback will be used")

from cmap_demo.header import print_header

print()
print_header(stage="Setup complete")

## Stage 1. Making text machine-readable

Three things are going on when a transcript becomes a table.

**One row per unit of talk, with stable identifiers.** The unit here is a
speaker turn. Each row carries a `doc_id` that does not change, so a code
attached to row 47 stays attached to row 47 through every later step.

**The transformation is reconstructible.** Each row also records where its text
started and ended in the normalized document. The rows are not a summary of the
transcript; the normalized transcript can be rebuilt from them. Cell 5 does
exactly that and checks the result character for character.

**The format is a fixed schema.** Twelve columns, the same ones the coded
corpus in Stage 2 uses, which is why rows this notebook produces and rows read
from that corpus are interchangeable.

| Column | Holds |
|---|---|
| `project` | Which body of data this row belongs to |
| `number` | The segment label; here, the speaker of the turn |
| `reference` | Position of the turn within the document |
| `text` | The normalized text of the turn |
| `document` | Which transcript the row came from |
| `start_position`, `end_position` | Character offsets in the normalized document |
| `data_group` | The kind of data, as a list: `['interview']` |
| `text_length`, `word_count` | Size of the text |
| `doc_id` | Stable unique row identifier |
| `codes` | The codes on this row, as a list |

Representing qualitative material as arrays so it can be analysed
computationally, while keeping it tied back to the original, is developed in
Abramson and Dohan (2015) and Abramson et al. (2018).

In [ ]:
# A raw transcript, exactly as it arrives. This one is synthetic, written for
# this repository so there is something messy to clean.

RAW_PATH = Path("data/raw/interview_demo01_20240115.txt")
raw_text = RAW_PATH.read_text()

print(f"{RAW_PATH}  ({len(raw_text):,} characters)")
print("-" * 72)
for line in raw_text.splitlines()[:15]:
    print(repr(line))

In [ ]:
# Filename to metadata, then clean the text.

from cmap_demo.normalize import (
    collapse_whitespace, detect_speaker, parse_filename, remove_timestamps,
)

metadata = parse_filename(RAW_PATH.name)
print("Filename convention: datasource_subject_date.txt")
print(f"  {RAW_PATH.name} -> {metadata}")
print()

# A non-conforming name fails loudly rather than guessing.
try:
    parse_filename("notes.txt")
except ValueError as error:
    print(f"Rejected as expected: {error}")
print()

print("Before and after, on the first three lines that carry a turn:")
print("-" * 72)
shown = 0
for line in raw_text.splitlines():
    if shown >= 3 or not line.strip():
        continue
    if remove_timestamps(line) == line and detect_speaker(line)[0] is None:
        continue  # front matter, nothing for this step to do
    cleaned = collapse_whitespace(remove_timestamps(line))
    speaker, remainder = detect_speaker(cleaned)
    print(f"  before : {line!r}")
    print(f"  after  : {cleaned!r}")
    print(f"  speaker: {speaker!r}")
    print(f"  text   : {remainder!r}")
    print()
    shown += 1

In [ ]:
# One row per speaker turn, in the CMAP schema, then validate it.

from cmap_demo.normalize import (
    build_cmap_frame, reconstruct_text, segment_turns, validate_cmap_frame,
)

turns = segment_turns(raw_text)
normalized = build_cmap_frame(turns, metadata)

normalized_path = OUTPUT_DIR / "normalized_sample.csv"
normalized.to_csv(normalized_path, index=False)
print(f"{len(turns)} turns -> {normalized_path}")

# Validation is not a column-name check. It confirms the required fields are
# non-empty, that codes and data_group round-trip to lists, that doc_id is
# unique, that word_count matches the text, and that the file re-reads to an
# identical frame.
validate_cmap_frame(normalized, csv_path=normalized_path)

# The reconstructibility claim from Stage 1, checked rather than asserted.
rebuilt = reconstruct_text(normalized)
expected = " ".join(normalized["text"])
print(f"Rebuilt {len(rebuilt):,} characters from the rows and their offsets; "
      f"identical to the normalized document: {rebuilt == expected}")
print()

normalized[["doc_id", "number", "word_count", "text"]].head()

## How the pieces fit together

*Figure placeholder: the author's pipeline diagram is inserted here.*

**This figure is an example from prior work, not from the data in this
notebook.** It comes from the ASA 2022 workshop run with Zhuofan Li and Daniel
Dohan ([materials](https://github.com/lizhuofan95/ASA2022_Workshop)), the
companion to Li, Dohan and Abramson (2021), *Socius* 7,
[doi:10.1177/23780231211062345](https://doi.org/10.1177/23780231211062345). Its
counts describe that corpus: 19,981 rows, of which 4,659 carry `#background`
and 15,322 do not.

The corpus vendored here is a different one. It holds **29,090 rows across 384
documents** in the same twelve-column format, and Stage 2 below prints those
numbers from the file itself rather than asking you to take them on trust.

What carries over is the shape of the work, not the numbers: interviews become
transcripts, transcripts become rows, and rows carry codes.


## De-identification: where it goes, and why it is skipped here

In a production pipeline de-identification goes here, immediately after
normalization, because everything downstream inherits whatever it lets through.

Follow your field's professional norms and your institutional review protocol.
Current workflows increasingly run de-identification tools before anything is
shared, and our teams run fully local models in secure systems for it (Abramson,
Prendergast, Li and Dohan 2026). Check with your IRB.

The step is skipped here because the IEEE History Center published these oral
histories under the interviewees' own names, so the corpus is already public.
Public is not the same as de-identified: the first row in Stage 2 carries a
name, an interviewer, a date, and a city. That is a property of this corpus,
not a shortcut to copy.


## Stage 2. A coded corpus in the same format

`data/1_cleaned_data.csv` holds engineering oral histories in the twelve-column
format Stage 1 just produced. The provenance chain runs:

**IEEE History Center**, the Engineering and Technology History Wiki, where
these oral histories were published → **ASA 2022 workshop**, run with
Zhuofan Li and Daniel Dohan, whose materials brought the corpus into a coded,
tabular form →
**CMAP format**, the shape used here and in the CMAP Visualization Toolkit.

Because it shares the schema, everything below would work the same way on rows
this notebook made itself.

> The data carries its own restriction and is **not** covered by this
> repository's BSD licence. No part of it may be quoted for publication
> without written permission from the Director of the IEEE History Center.
> See [`SAMPLE_DATA.md`](../SAMPLE_DATA.md).

In [ ]:
# Load the coded corpus and take a workable subset.

corpus = pd.read_csv("data/1_cleaned_data.csv")

print(f"rows      : {len(corpus):,}")
print(f"documents : {corpus['document'].nunique()}")
print(f"columns   : {len(corpus.columns)}")
print(f"            {list(corpus.columns)}")
print()

# Most rows are very short: the median turn is 16 words, and plenty are a
# single word. Keep rows with something to analyse, from the first 40
# documents, so the figures below are quick.
demo = corpus[corpus["word_count"] >= 25]
first_documents = demo["document"].unique()[:40]
demo = demo[demo["document"].isin(first_documents)].copy()

print(f"demo subset: {len(demo):,} rows across {demo['document'].nunique()} documents")
demo[["doc_id", "document", "word_count", "text"]].head(3)

## Stage 3. Three ways to code text at scale

Before the lanes, the map. This is the published table of ways to index
qualitative material, reproduced unchanged.

**Table 3, Qualitative indexing approaches and uses.** From Abramson,
Prendergast, Li and Dohan (2026), "Qualitative Research in an Era of Artificial
Intelligence," *Annual Review of Sociology* 52:20.1 to 20.27, p. 20.17. Open
access: [doi:10.1146/annurev-soc-011824-104836](https://doi.org/10.1146/annurev-soc-011824-104836).

| Approach | When to use | Purpose/strength | Cautions/limits |
|---|---|---|---|
| **List/dictionary indexing** | When tagging well-defined, high-certainty concepts with clear linguistic boundaries (e.g., clinical trials, transportation_mention) | Enables rapid, consistent tagging across large volumes of text<br>Supports fully reproducible search and comparison | Risks missing nuance or context<br>False positives<br>Should be limited to terms with stable meaning<br>Use prefix in QDA software (e.g., auto_symptom) to flag for review |
| **Full human coding** | When analyzing interpretive, context-dependent, or emergent categories (e.g., age_stratification, illness_stigma) | Preserves contextual richness and interpretive depth<br>Essential for meaning-centered or theory-building work | Time-intensive<br>Intercoder reliability and review needed to ensure consistency |
| **Hybrid: human-machine learning** | When extending researcher-defined codes (e.g., social_network_mentions, health_disparity) across very large corpora | Scales human classifications through supervised learning<br>Combines automation with iterative review<br>Reduces repetitive tagging | Requires quality human training data and setup<br>Models can drift or amplify bias if not used well<br>Secondary human review essential |
| **Pattern discovery / modeling** | When exploring co-occurrence, conceptual relationships, or latent meaning across and within text data (e.g., word embeddings, semantic networks, topic modeling) | Reveals associations and structures not immediately recognized<br>Useful for visualizations and confirming patterns | Results are probabilistic and must be interpreted in dialog with qualitative data<br>Always verify patterns by returning to source text |

> These approaches can be combined in a project, such as using lists to index
> basic terms or segments, machine learning to tag more complex topics, and
> human coding to classify deeper distinctions. Abbreviation: QDA, qualitative
> data analysis.

That closing note is this notebook's own structure, which is why the lanes below
are not presented as alternatives to each other.

**Where this notebook sits in the table.** Lane 1 is list/dictionary indexing.
Lane 2 is the hybrid human-machine learning row. The figures in Stage 4 are
pattern discovery and modeling. Full human coding has no lane here; it is the
human baseline everything else is measured against.

**Lane 3, the large language model lane, has no row of its own.** It is an
additional lane in this notebook that the table's four rows do not directly
represent.

**The concepts this demo uses.** One vocabulary of four concepts, applied in
every lane below so the three sets of results are comparable.

| Concept | What the dictionary looks for |
|---|---|
| `career` | job, jobs, career, work, worked, working, employ*, hired, position, company, laborator*, promot*, salary, colleague, retire*, profession* |
| `education` | school, college, univers*, degree, professor, student, studi*, class, classes, classroom, teach*, educat*, graduat*, thesis, undergraduate, doctorate |
| `family` | father, mother, dad, mom, parent, parents, brother, sister, wife, husband, son, daughter, child, children, famil*, grandfather, grandmother, marri*, uncle, aunt, cousin |
| `military` | army, navy, air force, marines, military, war, wartime, enlist*, soldier, veteran, combat, corps, battalion, draft, drafted |

An asterisk marks a stem: `educat*` matches education, educated, educational.

**A dictionary.** You write the patterns; the machine applies them. Completely
transparent and completely reproducible, and it only ever finds what you
thought to look for. Always read the matches; a term that looks obvious often
fires on something you did not intend.

**A model.** Segments are embedded and a classifier is trained on labelled
examples. Handles phrasing a dictionary would miss, needs labelled data, and
what it learned is much harder to inspect.

**A large language model.** Instructions in prose instead of patterns or
training data. Fast to start and easy to get wrong: outputs drift between runs,
and a model will happily produce confident output about text it never read.

They are not rivals. The dictionary gives you a transparent floor, the model
scales it, and the language model handles what neither anticipated, and the
hybrid human-machine combination is where the useful work tends to happen
(Li, Dohan and Abramson 2021).

In [ ]:
# Lane 1. Dictionary and regex, with real matches on the real corpus.

from collections import Counter

from cmap_demo.llm_handoff import (
    ALLOWED_CODES, code_by_dictionary, codes_to_source_shape, dictionary_matches,
)

# The four concepts and the words behind them are in the table above. The
# patterns themselves live in cmap_demo/llm_handoff.py, which is the file to
# edit if you want a different vocabulary.
dictionary_codes = [code_by_dictionary(text) for text in demo["text"]]
counts = Counter(code for row in dictionary_codes for code in row)

print(f"Rows matched, out of {len(demo):,} in the demo subset:")
print(f"  {'concept':<10} {'rows':>6} {'share':>7}")
for code_name in ALLOWED_CODES:
    n = counts[code_name]
    print(f"  {code_name:<10} {n:>6} {n / len(demo):>7.0%}")

# A dictionary that matches nothing is a broken dictionary, not an empty corpus.
empty = [c for c in ALLOWED_CODES if counts[c] == 0]
if empty:
    raise RuntimeError(f"These dictionaries matched nothing: {empty}. Fix them.")

# Counts are worth nothing until you have read what produced them. The corpus
# notice asks that we quote sparingly, so this prints three segments and cuts
# them short, and it skips the five rows the language model lane prints later.
llm_sample = set(demo.head(5)["doc_id"])

print("\nThree matched segments, cut short:")
shown = 0
for _, row in demo.iterrows():
    hits = dictionary_matches(row["text"])
    if not hits or shown >= 3 or row["doc_id"] in llm_sample:
        continue
    text = " ".join(str(row["text"]).split())
    excerpt = text[:200] + ("..." if len(text) > 200 else "")
    print(f"\n  doc_id {row['doc_id']}  {sorted({c for c, _ in hits})}")
    print(f"  {excerpt}")
    shown += 1

# These are the codes the rest of the notebook uses. Write them in the same
# stringified-list shape the corpus uses, which is what lets the recycled
# toolkit figures below read the column unmodified.
demo["codes"] = codes_to_source_shape(dictionary_codes)

coded_path = OUTPUT_DIR / "coded_demo.csv"
demo.to_csv(coded_path, index=False)
validate_cmap_frame(demo, csv_path=coded_path)
print(f"\nWrote {coded_path}")

demo[["doc_id", "word_count", "codes"]].head(3)

## Lane 2. The model lane

This is the lane that needs a transformer, labelled examples, and a training
step, so this notebook links it rather than shipping a model download that
would stall a live demo.

- **[The ASA 2022 workshop](https://github.com/lizhuofan95/ASA2022_Workshop)**,
  run with Zhuofan Li and Daniel Dohan and based on the *Socius* paper cited
  below, gives the full transformer walkthrough on this same corpus, with a
  runnable
  [Colab notebook](https://colab.research.google.com/drive/1qMwvjaY6DKQ-jxFTyXt3S3qNQdpV_S9n)
- **[CMAP Visualization Toolkit](https://github.com/Computational-Ethnography-Lab/cmap_visualization_toolkit)**
  covers embeddings, clustering, t-SNE, and semantic networks over coded corpora
- **[Using machine learning with ethnographic interviews](https://cmabramson.com/resources/f/using-machine-learning-with-ethnographic-interviews)**
  is the author's methods-resources page

The result behind this lane is that human and machine coding combined
outperform either alone, reported in Li, Dohan and Abramson (2021),
"Qualitative Coding in the Computational Era," *Socius* 7
([doi:10.1177/23780231211062345](https://doi.org/10.1177/23780231211062345)).
That finding is theirs and is cited here, not recomputed below.

The figures in the next cell come from the CMAP Visualization Toolkit and show
what this class of output looks like at full scale.

In [ ]:
# Two figures from the CMAP Visualization Toolkit (BSD 3-Clause), showing what
# the full toolkit produces. These are the toolkit's own published figures,
# copied byte for byte from its documentation, not generated here.

from IPython.display import Image, display

for caption, path in [
    ("Code co-occurrence heatmap, full toolkit", "docs/cmap_heatmap.png"),
    ("Semantic network, full toolkit", "docs/cmap_semantic_network.png"),
]:
    print(f"{caption}  ({path})")
    display(Image(filename=path, width=620))

print("Source: CMAP Visualization Toolkit, BSD 3-Clause,")
print("https://github.com/Computational-Ethnography-Lab/cmap_visualization_toolkit")

## Lane 3. The large language model lane

The notebook never calls a model. It prints a prompt, you run it wherever you
already work, and you bring the answer back. No keys, no endpoints, and
nothing leaves this machine on its own.

The prompt closes off the ways this fails. It asks for CSV and forbids prose,
because a model given room to explain itself will take it, and prose wrapped
around a table is something you then strip back off by hand. It states the
allowed codes as a closed list, and it requires one row out for every row in.
Splitting and validating happen afterward in code, where they are
reproducible.

One rule does more work than the rest: each row's `doc_id` goes out with it and
has to come back unchanged. A model with no access to the data can return five
well-formed rows it invented outright. Parsing proves syntax, not grounding.
Checking the returned identifiers against the local source is what separates an
answer about your rows from an answer about nothing, and `check_llm_response`
in `cmap_demo/llm_handoff.py` does exactly that.

Be precise about what the check buys. It proves the answer is about the rows
you sent. It does not prove the codes are right, because an answer that returns
your `doc_id`s and labels every row `family` passes it as cleanly as a careful
one does. Reading the text is still your job.

One caveat, which the prompt states too. Each row is cut to about 300
characters before it goes out, because this corpus restricts quotation. A model
that cannot open the public URL will never see evidence that falls past the
cutoff.

In [ ]:
# Build the prompt. Copy everything between the rules into your assistant, run
# it there, and read the answer against the shape below before you use it.

from cmap_demo.llm_handoff import PUBLIC_CSV_URL, build_llm_prompt

llm_rows = demo.head(5)
prompt = build_llm_prompt(llm_rows, n_rows=5, csv_url=PUBLIC_CSV_URL)

print("=" * 72)
print(prompt)
print("=" * 72)

print(f"\nFive rows went out, so five rows come back, carrying these doc_ids: "
      f"{[int(i) for i in llm_rows['doc_id']]}.")
print("Here is a real answer to this prompt, kept in "
      "docs/llm_reference_output.csv:")

pd.read_csv("docs/llm_reference_output.csv")

## Stage 4. Visualize

Two figures, both drawn from the codes this notebook just produced rather than
from anything shipped with the corpus. The code behind them is recycled
directly from the CMAP Visualization Toolkit, so what you see here is a
simplified version of the real thing rather than a lookalike.

These are the simple examples. The
[full toolkit](https://github.com/Computational-Ethnography-Lab/cmap_visualization_toolkit)
has the interactive versions: embeddings, clustering, t-SNE projections, and
semantic networks, along with its own Colab notebook.

In [ ]:
# Figure 1. Word cloud over the demo subset.

from cmap_demo.viz import generate_wordcloud

# If you raised the word_count threshold where the demo subset is built in
# Stage 2, the subset can end up empty or nearly so. The underlying library
# raises a bare "We need at least 1 word to plot a word cloud" in that case,
# so say something useful first.
usable = demo["text"].dropna().astype(str).str.strip()
usable = usable[usable != ""]
if len(usable) == 0:
    raise ValueError(
        "The demo subset has no text left to plot. Lower the word_count "
        "threshold where the subset is built in Stage 2, or widen the "
        "document selection, then re-run from there through the dictionary "
        "lane before coming back here."
    )

generate_wordcloud(
    demo["text"],
    title="Engineering oral histories",
    out_dir=OUTPUT_DIR,
)

In [ ]:
# Figure 2. How often the notebook's own codes appear on the same row.
#
# This reads output/coded_demo.csv from disk rather than the frame in memory.
# So if you go back and change the subset in Stage 2, re-run the dictionary
# lane before this one; otherwise you get a figure of the PREVIOUS subset with
# no error to warn you.

from cmap_demo.viz import create_code_cooccurrence_heatmap

create_code_cooccurrence_heatmap(
    filepath=str(coded_path),
    num_codes=4,
    clustered=True,
    out_dir=OUTPUT_DIR,
)

## Where to go next

**The full toolkit.**
[CMAP Visualization Toolkit](https://github.com/Computational-Ethnography-Lab/cmap_visualization_toolkit)
(BSD 3-Clause) has the interactive versions of the figures above.
[CMAP QDPX Converter](https://github.com/Computational-Ethnography-Lab/cmap_qdpx_converter)
gets data out of ATLAS.ti, NVivo, or MAXQDA and into this format.

**Learning resources.**
The [AI wiki](https://github.com/Computational-Ethnography-Lab/ai-wiki) for
concepts and a glossary, and the
[teaching bibliography](https://github.com/Computational-Ethnography-Lab/teaching#v-bibliography)
for the curated reading list with DOIs. Both are kept current; this notebook
links them rather than duplicating them.

**Works cited here.**

- Abramson, Corey M., and Daniel Dohan. 2015. "Beyond Text: Using Arrays to
  Represent and Analyze Ethnographic Data." *Sociological Methodology*
  45(1):272–319. [doi:10.1177/0081175015578740](https://doi.org/10.1177/0081175015578740)
- Abramson, Corey M., Jacqueline Joslyn, Katharine A. Rendle, Sarah B. Garrett,
  and Daniel Dohan. 2018. "The Promises of Computational Ethnography."
  *Ethnography* 19(2):254–284.
  [doi:10.1177/1466138117725340](https://doi.org/10.1177/1466138117725340)
- Li, Zhuofan, Daniel Dohan, and Corey M. Abramson. 2021. "Qualitative Coding
  in the Computational Era." *Socius* 7.
  [doi:10.1177/23780231211062345](https://doi.org/10.1177/23780231211062345)
- Abramson, Corey M., Tara Prendergast, Zhuofan Li, and Daniel Dohan. 2026.
  "Qualitative Research in an Era of Artificial Intelligence." *Annual Review
  of Sociology* 52:20.1–20.27.
  [doi:10.1146/annurev-soc-011824-104836](https://doi.org/10.1146/annurev-soc-011824-104836)

The full list, including every repository and dataset used, is in
[`REFERENCES.md`](../REFERENCES.md).

**Licensing, once more.** The code is BSD 3-Clause. The dataset is not. It
keeps the IEEE History Center's restriction on quotation. The figures in the
model-lane cell belong to the CMAP Visualization Toolkit and are BSD 3-Clause.